# Test: `get_synonyms()` for FishBase

Run from the **project root** so that `scripts/` is on the Python path.

FishBase has no public REST/JSON API. This client scrapes two HTML pages:
- `/summary/{Genus}-{Species}` — resolves accepted name + SpecCode
- `/nomenclature/{SpecCode}` — synonym list with all fields in link parameters

Covers:
- Accepted name lookup
- Synonym lookup (verifies the summary page resolves synonyms to the accepted taxon)
- Name not in FishBase (expects empty list)

In [1]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from urllib.parse import quote
from IPython.display import Markdown, display as ipy_display

from scripts.APIs_pipe.fishbase import FishBaseAPI

In [2]:
# (label, species, notes)
TEST_CASES = [
    ("accepted name", "Gadus morhua",    "Atlantic cod — well-known fish with many synonyms"),
    ("synonym query", "Gadus callarias", "junior synonym of G. morhua — tests summary-page resolution"),
    ("not in FishBase", "Amanita muscaria", "terrestrial fungus — expects empty list"),
]

In [3]:
client = FishBaseAPI()

for label, species, notes in TEST_CASES:
    print(f"\n{'='*60}")
    print(f"  [{label}]")
    print(f"  Species: {species}")
    print(f"  Notes:   {notes}")
    print(f"{'='*60}")
    try:
        result = client.get_synonyms(species)
        print(f"Synonyms returned: {len(result)}")
        ipy_display(result)
    except Exception as e:
        print(f"ERROR: {e}")


  [accepted name]
  Species: Gadus morhua
  Notes:   Atlantic cod — well-known fish with many synonyms
Synonyms returned: 12


[{'name': 'Gadus morhua',
  'author': 'Linnaeus',
  'publication_year': '1758',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Asellus major',
  'author': 'Not given',
  'publication_year': '',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus callarias',
  'author': 'Linnaeus',
  'publication_year': '1758',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus vertagus',
  'author': 'Walbaum',
  'publication_year': '1792',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus heteroglossus',
  'author': 'Walbaum',
  'publication_year': '1792',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus ruber',
  'author': 'Lacepède',
  'publication_year': '1803',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'


  [synonym query]
  Species: Gadus callarias
  Notes:   junior synonym of G. morhua — tests summary-page resolution
Synonyms returned: 12


[{'name': 'Gadus morhua',
  'author': 'Linnaeus',
  'publication_year': '1758',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Asellus major',
  'author': 'Not given',
  'publication_year': '',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus callarias',
  'author': 'Linnaeus',
  'publication_year': '1758',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus vertagus',
  'author': 'Walbaum',
  'publication_year': '1792',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus heteroglossus',
  'author': 'Walbaum',
  'publication_year': '1792',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus ruber',
  'author': 'Lacepède',
  'publication_year': '1803',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'


  [not in FishBase]
  Species: Amanita muscaria
  Notes:   terrestrial fungus — expects empty list
Synonyms returned: 0


[]

---
## Internal pipeline steps

Walk through each method individually for `Gadus morhua`.

### Step 1: `_fetch_query_data`

Fetches `/summary/{Genus}-{Species}`. FishBase serves the accepted species page regardless of whether the name is accepted or a synonym — both the SpecCode and accepted name are embedded in the HTML.

In [4]:
from scripts.utils.normalize_strings import normalize_scientific_name

SPECIES = "Gadus morhua"
name = normalize_scientific_name(SPECIES)
parts = name.split()

url_summary = f"https://www.fishbase.se/summary/{parts[0]}-{parts[1]}"
ipy_display(Markdown(f"**URL:** [{url_summary}]({url_summary})"))

raw_data = client._fetch_query_data(name)
print(f"\nspec_code       : {raw_data.get('spec_code')}")
print(f"accepted_genus  : {raw_data.get('accepted_genus')}")
print(f"accepted_species: {raw_data.get('accepted_species')}")

**URL:** [https://www.fishbase.se/summary/Gadus-morhua](https://www.fishbase.se/summary/Gadus-morhua)


spec_code       : 69
accepted_genus  : Gadus
accepted_species: morhua


### Step 2: `_extract_internal_id`

Returns the SpecCode from the query result. Also shows the difference when querying a synonym — the summary page resolves `Gadus callarias` to the same SpecCode as `Gadus morhua`.

In [5]:
spec_code = client._extract_internal_id(raw_data)
print(f"SpecCode (accepted): {spec_code}")

url_accepted = f"https://www.fishbase.se/summary/Gadus-morhua"
ipy_display(Markdown(f"**Accepted summary page:** [{url_accepted}]({url_accepted})"))

# Show synonym resolution
raw_data_syn = client._fetch_query_data("Gadus callarias")
spec_code_syn = client._extract_internal_id(raw_data_syn)
print(f"\nSynonym query: Gadus callarias")
print(f"  accepted_genus  : {raw_data_syn.get('accepted_genus')}")
print(f"  accepted_species: {raw_data_syn.get('accepted_species')}")
print(f"  SpecCode        : {spec_code_syn}  ← same as G. morhua")

url_syn = f"https://www.fishbase.se/summary/Gadus-callarias"
ipy_display(Markdown(f"**Synonym summary page:** [{url_syn}]({url_syn}) ← FishBase serves G. morhua page"))

SpecCode (accepted): 69


**Accepted summary page:** [https://www.fishbase.se/summary/Gadus-morhua](https://www.fishbase.se/summary/Gadus-morhua)


Synonym query: Gadus callarias
  accepted_genus  : Gadus
  accepted_species: morhua
  SpecCode        : 69  ← same as G. morhua


**Synonym summary page:** [https://www.fishbase.se/summary/Gadus-callarias](https://www.fishbase.se/summary/Gadus-callarias) ← FishBase serves G. morhua page

### Step 3: `_fetch_synonym_data`

Fetches `/nomenclature/{SpecCode}`. Each synonym is an anchor tag with all fields encoded in its `SynonymSummary.php` href — GenusName, SpeciesName, Author, Status, Misspelling, SpecCode.

In [6]:
url_nom = f"https://www.fishbase.se/nomenclature/{spec_code}"
ipy_display(Markdown(f"**URL:** [{url_nom}]({url_nom})"))

synonym_data = client._fetch_synonym_data(raw_data)
print(f"\nRaw synonym parameter dicts: {len(synonym_data)}")
print(f"\nFirst record (all available fields):")
ipy_display(synonym_data[0])

**URL:** [https://www.fishbase.se/nomenclature/69](https://www.fishbase.se/nomenclature/69)


Raw synonym parameter dicts: 20

First record (all available fields):


{'ID': '29394',
 'GSID': '12302',
 'Status': 'accepted name',
 'Synonymy': 'senior synonym',
 'Combination': 'original combination',
 'GenusName': 'Gadus',
 'SpeciesName': 'morhua',
 'SpecCode': '69',
 'SynonymsRef': '1371',
 'Author': 'Linnaeus, 1758',
 'Misspelling': '0'}

### Step 4: `_compile_synonyms`

Filters out misspellings and subspecific names, splits `Author` into `author` + `publication_year`, and maps to the pipeline-standard five-key schema.

In [7]:
compiled = client._compile_synonyms(synonym_data)
print(f"Compiled records: {len(compiled)}")
ipy_display(compiled)

Compiled records: 12


[{'name': 'Gadus morhua',
  'author': 'Linnaeus',
  'publication_year': '1758',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Asellus major',
  'author': 'Not given',
  'publication_year': '',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus callarias',
  'author': 'Linnaeus',
  'publication_year': '1758',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus vertagus',
  'author': 'Walbaum',
  'publication_year': '1792',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus heteroglossus',
  'author': 'Walbaum',
  'publication_year': '1792',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'name': 'Gadus ruber',
  'author': 'Lacepède',
  'publication_year': '1803',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/69'},
 {'

---
## Edge case: species with few or no synonyms

`Danio rerio` (zebrafish) is a well-known fish expected to have few synonyms in FishBase.

In [8]:
no_syn_species = "Danio rerio"
url_no_syn = f"https://www.fishbase.se/summary/Danio-rerio"
ipy_display(Markdown(f"**Summary page:** [{url_no_syn}]({url_no_syn})"))

raw_no_syn = client._fetch_query_data(no_syn_species)
spec_no_syn = raw_no_syn.get("spec_code", "")
url_nom_no_syn = f"https://www.fishbase.se/nomenclature/{spec_no_syn}"
ipy_display(Markdown(f"**Nomenclature page:** [{url_nom_no_syn}]({url_nom_no_syn})"))

result = client.get_synonyms(no_syn_species)
print(f"\nSynonyms returned: {len(result)}")
ipy_display(result)

**Summary page:** [https://www.fishbase.se/summary/Danio-rerio](https://www.fishbase.se/summary/Danio-rerio)

**Nomenclature page:** [https://www.fishbase.se/nomenclature/4653](https://www.fishbase.se/nomenclature/4653)


Synonyms returned: 10


[{'name': 'Danio rerio',
  'author': '(Hamilton, 1822)',
  'publication_year': '',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/4653'},
 {'name': 'Cyprinus rerio',
  'author': 'Hamilton',
  'publication_year': '1822',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/4653'},
 {'name': 'Barilius rerio',
  'author': '(Hamilton, 1822)',
  'publication_year': '',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/4653'},
 {'name': 'Brachydanio rerio',
  'author': '(Hamilton, 1822)',
  'publication_year': '',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/4653'},
 {'name': 'Nuria rerio',
  'author': '(Hamilton, 1822)',
  'publication_year': '',
  'publication_name': 'U',
  'api_link': 'https://www.fishbase.se/nomenclature/4653'},
 {'name': 'Cyprinus chapalio',
  'author': 'Hamilton',
  'publication_year': '1822',
  'publication_name': 'U',
  'api_link': 'https://www.fis

---
## Alternative access: DuckDB + Parquet

FishBase data is also available as open Parquet files on S3 (via the rOpenSci / source.coop project). This gives direct SQL access to the raw database tables — no scraping needed.

> **Note:** The URL below is the working one (`data.source.coop/cboettig/…`). The `us-west-2.opendata.source.coop/fishbase/…` pattern does not resolve.
>
> Requires: `pip install duckdb`

In [9]:
import duckdb

con = duckdb.connect()
BASE_PARQUET = "https://data.source.coop/cboettig/fishbase/fb/v24.07/parquet"

# --- Step 1: look up SpecCode from the species table ---
species_df = con.execute(f"""
    SELECT SpecCode, Genus, Species, Author, FBname
    FROM read_parquet('{BASE_PARQUET}/species.parquet')
    WHERE Genus = 'Gadus' AND Species = 'morhua'
""").df()

print("Species table row:")
ipy_display(species_df)

spec_code_duckdb = int(species_df["SpecCode"].iloc[0])
print(f"\nSpecCode: {spec_code_duckdb}")

Species table row:


,SpecCode,Genus,Species,Author,FBname
0,69,Gadus,morhua,"Linnaeus, 1758",Atlantic cod



SpecCode: 69


In [10]:
# --- Step 2: query synonyms table for that SpecCode ---
# TaxonLevel = 'Species' excludes subspecific entries.
# Misspelling = 0 excludes misspellings (same filters as FishBaseAPI._compile_synonyms).
# Year is a separate integer column here — unlike the HTML scraper which parses it from Author.
synonyms_df = con.execute(f"""
    SELECT SynGenus, SynSpecies, Author, Year, Status, Synonymy, Misspelling
    FROM read_parquet('{BASE_PARQUET}/synonyms.parquet')
    WHERE SpecCode = {spec_code_duckdb}
      AND TaxonLevel = 'Species'
      AND Misspelling = 0
    ORDER BY StatusOrder
""").df()

print(f"Synonyms returned: {len(synonyms_df)}")
ipy_display(synonyms_df)

Synonyms returned: 12


,SynGenus,SynSpecies,Author,Year,Status,Synonymy,Misspelling
0,Gadus,morhua,"Linnaeus, 1758",1758,accepted name,senior synonym,0
1,Asellus,major,Not given,<NA>,ambiguous synonym,other,0
2,Gadus,callarias,"Linnaeus, 1758",1758,synonym,junior synonym,0
3,Gadus,vertagus,"Walbaum, 1792",1792,synonym,junior synonym,0
4,Gadus,heteroglossus,"Walbaum, 1792",1792,synonym,junior synonym,0
5,Gadus,ruber,"Lacepède, 1803",1803,ambiguous synonym,questionable,0
6,Gadus,arenosus,"Mitchill, 1815",1815,synonym,junior synonym,0
7,Gadus,rupestris,"Mitchill, 1815",1815,synonym,junior synonym,0
8,Morhua,vulgaris,"Fleming, 1828",1828,synonym,junior synonym,0
9,Morhua,punctatus,"Fleming, 1828",1828,synonym,junior synonym,0


### One-shot join: name → synonyms without a two-step lookup

You can also join `species` and `synonyms` directly, skipping the SpecCode lookup entirely.

In [11]:
joined_df = con.execute(f"""
    SELECT syn.SynGenus, syn.SynSpecies, syn.Author, syn.Year, syn.Status
    FROM read_parquet('{BASE_PARQUET}/species.parquet')  AS sp
    JOIN read_parquet('{BASE_PARQUET}/synonyms.parquet') AS syn
      ON sp.SpecCode = syn.SpecCode
    WHERE sp.Genus   = 'Gadus'
      AND sp.Species = 'morhua'
      AND syn.TaxonLevel  = 'Species'
      AND syn.Misspelling = 0
    ORDER BY syn.StatusOrder
""").df()

print(f"Synonyms returned: {len(joined_df)}")
ipy_display(joined_df)

Synonyms returned: 12


,SynGenus,SynSpecies,Author,Year,Status
0,Gadus,morhua,"Linnaeus, 1758",1758,accepted name
1,Asellus,major,Not given,<NA>,ambiguous synonym
2,Gadus,callarias,"Linnaeus, 1758",1758,synonym
3,Gadus,vertagus,"Walbaum, 1792",1792,synonym
4,Gadus,heteroglossus,"Walbaum, 1792",1792,synonym
5,Gadus,ruber,"Lacepède, 1803",1803,ambiguous synonym
6,Gadus,arenosus,"Mitchill, 1815",1815,synonym
7,Gadus,rupestris,"Mitchill, 1815",1815,synonym
8,Morhua,vulgaris,"Fleming, 1828",1828,synonym
9,Morhua,punctatus,"Fleming, 1828",1828,synonym
